# Modeling & Training

## General Description

This notebook focuses on developing, training, and evaluating image classification models capable of identifying rice leaf diseases from images.

The notebook uses the processed datasets generated during preprocessing to train convolutional neural network (CNN) models for multi-class disease classification. Different modeling approaches and training configurations will be explored to identify a reliable and efficient solution for rice leaf disease prediction.

The notebook also includes model evaluation, performance visualization, and preparation of final artifacts for deployment in the Streamlit dashboard.


## Objectives

- Prepare TensorFlow image generators for model training.
- Build and train image classification models.
- Evaluate model performance on validation data.
- Analyze classification accuracy and prediction behavior.
- Compare training and validation performance.
- Identify potential overfitting or underfitting.
- Save trained models and evaluation outputs for deployment.


## Inputs

Processed datasets and metadata generated during preprocessing:

- `inputs/datasets/processed/train_labels.csv`
- `inputs/datasets/processed/val_labels.csv`

Raw image dataset:
- `inputs/datasets/raw/rice/`

Preprocessing outputs:
- `outputs/datasets/preprocessing/preprocessing_summary.csv`


## Outputs

Generated model artifacts and evaluation outputs saved to:

`outputs/models/`

Including:
- trained classification model
- training history
- evaluation metrics
- confusion matrix visualizations
- prediction examples


## Additional Comments

- Transfer learning will be considered to improve model performance and reduce training time.
- Dataset imbalance may affect classification accuracy for minority disease classes.
- Data augmentation strategies defined during preprocessing will be applied during training.
- Model performance will be evaluated using both quantitative metrics and visual analysis.

### 1. Load Python Packages and Project Dependencies

This section imports the libraries and dependencies required for image classification model development, training, evaluation, and visualization.

In [1]:
# Data handling
import pandas as pd
import numpy as np

# File and path handling
from pathlib import Path
import os
import json

# Visualisation
import matplotlib.pyplot as plt

# TensorFlow / Keras
import tensorflow as tf

from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator
)

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.applications import (
    MobileNetV2
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

from tensorflow.keras.optimizers import Adam

# Evaluation
from sklearn.metrics import accuracy_score

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


In [2]:
def setup_root():
    current_path = Path().resolve()
    # Search upwards for project root
    while not (current_path / "requirements.txt").exists():
        if current_path == current_path.parent:
            raise FileNotFoundError(
                "Project root not found."
            )
        current_path = current_path.parent
    return current_path


# Set project root
PROJECT_ROOT = setup_root()

# Change working directory
os.chdir(PROJECT_ROOT)
print(f"Project root set to: {PROJECT_ROOT}")


# Define paths
RAW_DATA_DIR = Path(
    "inputs/datasets/raw/rice"
)

PROCESSED_DATA_DIR = Path(
    "inputs/datasets/processed"
)

MODEL_OUTPUT_DIR = Path(
    "outputs/models"
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Processed dataset directory: "
      f"{PROCESSED_DATA_DIR}")

print(f"Model output directory: "
      f"{MODEL_OUTPUT_DIR}")

Project root set to: C:\code\ml\workspace\rice_leaf_diseases_analyser
Processed dataset directory: inputs\datasets\processed
Model output directory: outputs\models


### 3. Load Processed Datasets

Load the processed training and validation metadata generated during preprocessing. These datasets contain image paths and classification labels used during model training.

In [4]:
# Load processed datasets
train_labels_path = (
    PROCESSED_DATA_DIR / "train_labels.csv"
)

val_labels_path = (
    PROCESSED_DATA_DIR / "val_labels.csv"
)

df_train = pd.read_csv(train_labels_path)

df_val = pd.read_csv(val_labels_path)

print("Datasets loaded successfully.")

print(f"\nTraining samples: {len(df_train)}")
print(f"Validation samples: {len(df_val)}")

Datasets loaded successfully.

Training samples: 6935
Validation samples: 1730


In [5]:
# Preview training dataset
display(df_train.head())

# Preview validation dataset
display(df_val.head())

,image_path,class_id,class_name
0,inputs\datasets\raw\rice\images\train\r6k_test...,6,LeafSmut
1,inputs\datasets\raw\rice\images\train\r6k_test...,6,LeafSmut
2,inputs\datasets\raw\rice\images\train\r6k_test...,6,LeafSmut
3,inputs\datasets\raw\rice\images\train\r6k_test...,6,LeafSmut
4,inputs\datasets\raw\rice\images\train\r6k_test...,6,LeafSmut


,image_path,class_id,class_name
0,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
1,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
2,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
3,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut
4,inputs\datasets\raw\rice\images\val\r6k_test_B...,6,LeafSmut


In [6]:
print("Training class distribution:\n")

print(
    df_train["class_name"]
    .value_counts()
)

print("\nValidation class distribution:\n")

print(
    df_val["class_name"]
    .value_counts()
)

Training class distribution:

class_name
LeafSmut               1600
BacterialLeafBlight    1600
BrownSpot              1600
LeafScald               365
NeckBlast               363
LeafBlast               360
NarrowBrownLeafSpot     355
Healthy                 347
Hispa                   345
Name: count, dtype: int64

Validation class distribution:

class_name
LeafSmut               400
BacterialLeafBlight    400
BrownSpot              400
LeafScald               91
NeckBlast               90
LeafBlast               89
NarrowBrownLeafSpot     88
Hispa                   86
Healthy                 86
Name: count, dtype: int64
